# Minimum Bayes Risk (MBR) Decoding — Exploration

**Status: exploratory trial, not part of the official AlexandriaX submission.**
(The submission pipeline uses standard beam search — see `models/infer.py`.)

This notebook sanity-checks MBR decoding on our fine-tuned NileChat-3B checkpoint
before deciding whether it was worth productionizing (see `models/infer.py --decoding mbr` for the reusable version).

Pipeline:
1. Generate `N` candidate translations per input via temperature + nucleus sampling.
2. Score every candidate against every other candidate with a utility function
   (sentence-level spBLEU, via `sacrebleu`).
3. Pick the candidate with the highest average utility against the rest of the
   pool — the MBR selection.
4. Compare against plain beam search on the same input.

The MBR logic itself (generate -> score -> select) is model-agnostic; only the
model loading and prompt format below are specific to this project.

## Setup & authentication

Run in Google Colab — pulls `HF_TOKEN` and `WANDB_API_KEY` from Colab Secrets.

In [1]:
!pip install sacrebleu -q
!pip install -U "bitsandbytes>=0.46.1" -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 27.9 MB/s eta 0:00:00


In [2]:
import os
from google.colab import userdata
from huggingface_hub import login

if "Colab-Alexandria" in os.environ or userdata.get("Colab-Alexandria"):
    hf_token = userdata.get("Colab-Alexandria")
    login(token=hf_token)

In [ ]:
import os
import wandb
from google.colab import userdata

try:
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    wandb.login()
except Exception:
    print(
        "Could not find your Colab Secret! "
        "Make sure the 'Secrets' tab (key icon on the left panel) "
        "has 'WANDB_API_KEY' set and notebook access enabled."
    )

## Imports

In [4]:
import itertools
from dataclasses import dataclass, replace
from pathlib import Path

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from sacrebleu.metrics import BLEU

## Config

In [5]:
@dataclass
class MBRConfig:
    model_name: str = "UBC-NLP/NileChat-3B"
    num_candidates: int = 20        # candidates sampled per input for MBR
    sampling_temperature: float = 0.8
    top_p: float = 0.9
    max_new_tokens: int = 60
    beam_width: int = 5             # for the beam-search baseline
    load_in_4bit: bool = True


CFG = MBRConfig()

## Load the fine-tuned checkpoint

Downloads the LoRA adapter from a W&B model artifact and applies it on top of
the base model, matching how it was trained.

In [ ]:
print(f"Loading model: {CFG.model_name} ...")

run = wandb.init(entity="RosettaAtAlexandriaX", project="DialectalArabicMT", job_type="inference")
artifact_path = "RosettaAtAlexandriaX/DialectalArabicMT/model-FullScaleFT-NCchat:v5"
artifact = run.use_artifact(artifact_path, type="model")
artifact_dir = Path(artifact.download())
print(f"Checkpoint downloaded to: {artifact_dir}")


def model_compute_dtype() -> torch.dtype:
    # fp16 (not bf16) - matches the T4-class GPUs this was trained/run on.
    return torch.float16


def build_quantization_config() -> BitsAndBytesConfig | None:
    if not CFG.load_in_4bit:
        return None
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=model_compute_dtype(),
        bnb_4bit_use_double_quant=True,
    )


adapter_config_path = artifact_dir / "adapter_config.json"
if not adapter_config_path.exists():
    raise FileNotFoundError(f"No LoRA adapter found at {artifact_dir}.")

tokenizer_source = artifact_dir if (artifact_dir / "tokenizer_config.json").exists() else CFG.model_name
tokenizer = AutoTokenizer.from_pretrained(tokenizer_source, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

base_model = AutoModelForCausalLM.from_pretrained(
    CFG.model_name,
    device_map="auto",
    torch_dtype=torch.float16,
    quantization_config=build_quantization_config(),
    trust_remote_code=True,
)

model = PeftModel.from_pretrained(base_model, artifact_dir)
if hasattr(model, "gradient_checkpointing_disable"):
    model.gradient_checkpointing_disable()
model.eval()
model.config.use_cache = True

print(f"Loaded fine-tuned model from: {artifact_dir}")

## Beam search baseline

In [7]:
def generate_beam_search(messages: list[dict], cfg: MBRConfig = CFG) -> str:
    """Standard beam search - single best hypothesis, for comparison against MBR."""
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            num_beams=cfg.beam_width,
            max_new_tokens=cfg.max_new_tokens,
            do_sample=False,
            early_stopping=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

## MBR utility function (sentence-level spBLEU)

In [8]:
try:
    # Fixed FLORES-200 tokenizer profile, matching the shared task's spBLEU metric.
    bleu = BLEU(tokenize="flores200", effective_order=True)
except Exception as exc:
    raise RuntimeError(
        "Could not initialize SacreBLEU's flores200 tokenizer. "
        "Install/upgrade sacrebleu and sentencepiece."
    ) from exc


def sentence_bleu_utility(hyp: str, ref: str) -> float:
    # Sentence-level spBLEU as a similarity/utility score (0-100).
    if hyp.strip() == "" or ref.strip() == "":
        return 0.0
    return round(bleu.sentence_score(hyp, [ref]).score, 6)

## Candidate generation (temperature + nucleus sampling)

In [9]:
def generate_candidates(messages: list[dict], cfg: MBRConfig = CFG) -> list[str]:
    """Generate a pool of `cfg.num_candidates` candidates via sampling."""
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            do_sample=True,
            num_beams=1,           # explicitly override the checkpoint's beam-search default
            temperature=cfg.sampling_temperature,
            top_p=cfg.top_p,
            max_new_tokens=cfg.max_new_tokens,
            num_return_sequences=cfg.num_candidates,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    return [
        tokenizer.decode(ids[inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        for ids in output_ids
    ]

## MBR selection

In [10]:
def mbr_select(candidates: list[str], utility_fn) -> tuple[str, dict]:
    """
    Pick the candidate that maximizes expected utility against the rest of
    the pool. Returns the selected candidate plus a dict of every candidate
    -> its average utility score, for inspection.
    """
    n = len(candidates)
    scores = [0.0] * n

    for i, j in itertools.permutations(range(n), 2):
        scores[i] += utility_fn(candidates[i], candidates[j])

    avg_scores = [s / (n - 1) if n > 1 else 0.0 for s in scores]

    ranked = sorted(zip(candidates, avg_scores), key=lambda x: x[1], reverse=True)
    best_candidate, _ = ranked[0]

    score_map = dict(zip(candidates, avg_scores))
    return best_candidate, score_map

## Compare beam search vs MBR on one input

In [11]:
def run(messages: list[dict], cfg: MBRConfig = CFG) -> dict:
    print(f"\nInput: {messages}\n")

    # Baseline
    beam_output = generate_beam_search(messages, cfg)
    print(f"[Beam search, width={cfg.beam_width}]\n  {beam_output}\n")

    # MBR
    print(f"Sampling {cfg.num_candidates} candidates "
          f"(T={cfg.sampling_temperature}, top_p={cfg.top_p}) ...")
    candidates = generate_candidates(messages, cfg)

    # De-duplicate while preserving order (avoids wasted pairwise scoring)
    seen = set()
    unique_candidates = []
    for c in candidates:
        if c not in seen:
            seen.add(c)
            unique_candidates.append(c)

    print(f"Generated {len(candidates)} candidates "
          f"({len(unique_candidates)} unique). Scoring pairwise ...")

    mbr_output, score_map = mbr_select(unique_candidates, sentence_bleu_utility)
    print(f"\n[MBR selection]\n  {mbr_output}\n")

    print("Candidates ranked by expected utility (MBR score):")
    ranked = sorted(score_map.items(), key=lambda x: x[1], reverse=True)
    for text, score in ranked:
        marker = " <-- selected" if text == mbr_output else ""
        print(f"  [{score:6.2f}] {text}{marker}")

    return {
        "input": messages,
        "beam_search": beam_output,
        "mbr": mbr_output,
        "all_candidates_scored": ranked,
    }

## Inspect on examples

A handful of representative conversations - different domains, dialects, and
with/without conversation history - to sanity-check MBR selection quality
against beam search.

In [12]:
SYSTEM_PROMPT = (
    "You are an expert translator.\n\n"
    "- Return only the translated text.\n"
    "- Do not add any code, explanations, comments, or any other extra text.\n"
    "- Keep the meaning and tone and respect the gender direction.\n"
    "- Consider the country, the domain, the participants, and the speaker in your translation.\n"
)

def system_msg():
    return {"content": SYSTEM_PROMPT, "role": "system"}

def user_msg(content):
    return {"content": content, "role": "user"}

**EG - Agriculture and farming - no history**

In [13]:
messages = [system_msg(), user_msg("Translate the English sentence into Egyptian Arabic (Cairene) Dialect.\n\n### Metadata:\n- Country: EG\n- Domain: Agriculture and farming\n- Participants: Farmer, Buyer\n- Speaker: Farmer\n- Speaker Direction: male -> male\n\n### Conversation History:\nNo previous turns (Start of conversation).\n\n### Sentence to Translate:\nGood morning. The best cucumbers and peppers you'll find today, fresh from the farm.")]
run(messages)
print("\n" + "=" * 70)


Input: [{'content': 'You are an expert translator.\n\n- Return only the translated text.\n- Do not add any code, explanations, comments, or any other extra text.\n- Keep the meaning and tone and respect the gender direction.\n- Consider the country, the domain, the participants, and the speaker in your translation.\n', 'role': 'system'}, {'content': "Translate the English sentence into Egyptian Arabic (Cairene) Dialect.\n\n### Metadata:\n- Country: EG\n- Domain: Agriculture and farming\n- Participants: Farmer, Buyer\n- Speaker: Farmer\n- Speaker Direction: male -> male\n\n### Conversation History:\nNo previous turns (Start of conversation).\n\n### Sentence to Translate:\nGood morning. The best cucumbers and peppers you'll find today, fresh from the farm.", 'role': 'user'}]

[Beam search, width=5]
  صباح الخير. احسن خيار وفلفل هتلاقيهم النهاردة، طازة من المزرعة.

Sampling 20 candidates (T=0.8, top_p=0.9) ...
Generated 20 candidates (15 unique). Scoring pairwise ...

[MBR selection]
  ص

**EG - Agriculture and farming - with history**

In [14]:
messages = [system_msg(), user_msg('Translate the English sentence into Egyptian Arabic (Cairene) Dialect.\n\n### Metadata:\n- Country: EG\n- Domain: Agriculture and farming\n- Participants: Seed Supplier, Market Organizer\n- Speaker: Seed Supplier\n- Speaker Direction: male -> male\n\n### Conversation History:\nSeed Supplier: Good morning, my friend. Is this spot by the main entrance available? It would be great for business.\nTranslation: صباح الخير يا صاحبي. المكان اللي عند المدخل الرئيسي فاضي؟ هيشتغل معايا حلو.\nMarket Organizer: Good morning to you. That one is already taken, but the corner spot over there is free. It gets a lot of foot traffic too.\nTranslation: صباح النور. ده محجوز، بس الركن هناك فاضي وبيعدي عليه ناس كتير.\n\n### Sentence to Translate:\nAlright, the corner spot it is. God willing, it will be a profitable day for everyone.')]
run(messages)
print("\n" + "=" * 70)


Input: [{'content': 'You are an expert translator.\n\n- Return only the translated text.\n- Do not add any code, explanations, comments, or any other extra text.\n- Keep the meaning and tone and respect the gender direction.\n- Consider the country, the domain, the participants, and the speaker in your translation.\n', 'role': 'system'}, {'content': 'Translate the English sentence into Egyptian Arabic (Cairene) Dialect.\n\n### Metadata:\n- Country: EG\n- Domain: Agriculture and farming\n- Participants: Seed Supplier, Market Organizer\n- Speaker: Seed Supplier\n- Speaker Direction: male -> male\n\n### Conversation History:\nSeed Supplier: Good morning, my friend. Is this spot by the main entrance available? It would be great for business.\nTranslation: صباح الخير يا صاحبي. المكان اللي عند المدخل الرئيسي فاضي؟ هيشتغل معايا حلو.\nMarket Organizer: Good morning to you. That one is already taken, but the corner spot over there is free. It gets a lot of foot traffic too.\nTranslation: صباح 

**EG - Legal and financial - turn 1, no history**

In [15]:
messages = [system_msg(), user_msg("Translate the English sentence into Egyptian Arabic (Cairene) Dialect.\n\n### Metadata:\n- Country: EG\n- Domain: Legal and financial\n- Participants: Client - Real Estate Agent\n- Speaker: Client\n- Speaker Direction: female -> male\n\n### Conversation History:\nNo previous turns (Start of conversation).\n\n### Sentence to Translate:\nHow can I be absolutely sure the current owner doesn't have any outstanding maintenance fees or installments with the developer?")]
run(messages)
print("\n" + "=" * 70)


Input: [{'content': 'You are an expert translator.\n\n- Return only the translated text.\n- Do not add any code, explanations, comments, or any other extra text.\n- Keep the meaning and tone and respect the gender direction.\n- Consider the country, the domain, the participants, and the speaker in your translation.\n', 'role': 'system'}, {'content': "Translate the English sentence into Egyptian Arabic (Cairene) Dialect.\n\n### Metadata:\n- Country: EG\n- Domain: Legal and financial\n- Participants: Client - Real Estate Agent\n- Speaker: Client\n- Speaker Direction: female -> male\n\n### Conversation History:\nNo previous turns (Start of conversation).\n\n### Sentence to Translate:\nHow can I be absolutely sure the current owner doesn't have any outstanding maintenance fees or installments with the developer?", 'role': 'user'}]

[Beam search, width=5]
  إزاي أكون متأكدة تماما إن المالك الحالي ملوش أي مصاريف صيانة مستحقة أو أقساط مع المطور؟

Sampling 20 candidates (T=0.8, top_p=0.9) ...

**EG - Legal and financial - turn 2, with history**

In [16]:
messages = [system_msg(), user_msg("Translate the English sentence into Egyptian Arabic (Cairene) Dialect.\n\n### Metadata:\n- Country: EG\n- Domain: Legal and financial\n- Participants: Client - Real Estate Agent\n- Speaker: Client\n- Speaker Direction: female -> male\n\n### Conversation History:\nClient: How can I be absolutely sure the current owner doesn't have any outstanding maintenance fees or installments with the developer?\nTranslation: إزاي أتأكد تماما من إن المالك الحالي مش عليه أي فلوس صيانة أو اقساط للسمسار؟\nAgent: It's a standard procedure. We will request a formal 'financial clearance letter' from the developer's office. This document officially confirms the seller has a zero balance.\nTranslation: ده إجراء روتيني، هنطلب خطاب رسمي لبراءة الذمة المالية من مكتب السمسار، الورقة الرسمية دي بتثبت إن رصيد البايع صفر.\n\n### Sentence to Translate:\nAnd the developer will issue this to me or the seller?")]
run(messages)
print("\n" + "=" * 70)


Input: [{'content': 'You are an expert translator.\n\n- Return only the translated text.\n- Do not add any code, explanations, comments, or any other extra text.\n- Keep the meaning and tone and respect the gender direction.\n- Consider the country, the domain, the participants, and the speaker in your translation.\n', 'role': 'system'}, {'content': "Translate the English sentence into Egyptian Arabic (Cairene) Dialect.\n\n### Metadata:\n- Country: EG\n- Domain: Legal and financial\n- Participants: Client - Real Estate Agent\n- Speaker: Client\n- Speaker Direction: female -> male\n\n### Conversation History:\nClient: How can I be absolutely sure the current owner doesn't have any outstanding maintenance fees or installments with the developer?\nTranslation: إزاي أتأكد تماما من إن المالك الحالي مش عليه أي فلوس صيانة أو اقساط للسمسار؟\nAgent: It's a standard procedure. We will request a formal 'financial clearance letter' from the developer's office. This document officially confirms th

**EG - Education and academia - with history**

In [17]:
messages = [system_msg(), user_msg("Translate the English sentence into Egyptian Arabic (Cairene) Dialect.\n\n### Metadata:\n- Country: EG\n- Domain: Education and academia\n- Participants: Student, Supervisor\n- Speaker: Student\n- Speaker Direction: female -> male\n\n### Conversation History:\nStudent: Doctor, to be honest, I'm feeling completely overwhelmed. I look at the remaining work and I feel like I'll never finish.\nTranslation: يا دكتور انا بصراحة حاسة إن الحمل كبير أوي عليا، كل ما أبص على الشغل بحس ان عمري ما بخلص\nSupervisor: Listen, this feeling is a normal part of the process. Every student feels this way at some point. You have done very strong work to get to this stage.\nTranslation: بصي، دي حاجة طبيعي انك تحسيها، كل طالب بيحس كدة في وقت معين. انتي عملتي شغل جامد جدا عشان توصلي للمرحلة دي\n\n### Sentence to Translate:\nI just feel so much pressure from my family and from myself to finish.")]
run(messages)
print("\n" + "=" * 70)


Input: [{'content': 'You are an expert translator.\n\n- Return only the translated text.\n- Do not add any code, explanations, comments, or any other extra text.\n- Keep the meaning and tone and respect the gender direction.\n- Consider the country, the domain, the participants, and the speaker in your translation.\n', 'role': 'system'}, {'content': "Translate the English sentence into Egyptian Arabic (Cairene) Dialect.\n\n### Metadata:\n- Country: EG\n- Domain: Education and academia\n- Participants: Student, Supervisor\n- Speaker: Student\n- Speaker Direction: female -> male\n\n### Conversation History:\nStudent: Doctor, to be honest, I'm feeling completely overwhelmed. I look at the remaining work and I feel like I'll never finish.\nTranslation: يا دكتور انا بصراحة حاسة إن الحمل كبير أوي عليا، كل ما أبص على الشغل بحس ان عمري ما بخلص\nSupervisor: Listen, this feeling is a normal part of the process. Every student feels this way at some point. You have done very strong work to get to

**MR - Professional and workplace - Hassaniya, with history**

In [18]:
messages = [system_msg(), user_msg("Translate the English sentence into Mauritanian Hassaniya Dialect.\n\n### Metadata:\n- Country: MR\n- Domain: Professional and workplace\n- Participants: Intern, Friendly Colleague\n- Speaker: Intern\n- Speaker Direction: male -> female\n\n### Conversation History:\nIntern: Hello, Mariem. I hope you are well. I was hoping you could help me. Who is the gentleman that manages the logistics team? I see him often but don't know his name.\nTranslation: مرحب مريم ، ان شاء الله اتعودي اتعودي ابخير ، كنت اندور اتساعديني، منه المسؤول عن الخدمات اللوجستية ؟ دائما انشوفو يغير ما نعرف اسمو\nFriendly Colleague: I am well, thank God. That is Mr. Brahim. He is a very kind man and has been with us for many years. He is in charge of all our shipping operations.\nTranslation: بخير الحمد لله ، ذاك إبراهيم ، راجل متعدل ماشاء الله ، او يشتقل هون لو سنوات او هو اللي مسؤول عن خدمات الشحن كاملة عندنا\n\n### Sentence to Translate:\nThank you, that is very helpful to know.")]
run(messages)
print("\n" + "=" * 70)


Input: [{'content': 'You are an expert translator.\n\n- Return only the translated text.\n- Do not add any code, explanations, comments, or any other extra text.\n- Keep the meaning and tone and respect the gender direction.\n- Consider the country, the domain, the participants, and the speaker in your translation.\n', 'role': 'system'}, {'content': "Translate the English sentence into Mauritanian Hassaniya Dialect.\n\n### Metadata:\n- Country: MR\n- Domain: Professional and workplace\n- Participants: Intern, Friendly Colleague\n- Speaker: Intern\n- Speaker Direction: male -> female\n\n### Conversation History:\nIntern: Hello, Mariem. I hope you are well. I was hoping you could help me. Who is the gentleman that manages the logistics team? I see him often but don't know his name.\nTranslation: مرحب مريم ، ان شاء الله اتعودي اتعودي ابخير ، كنت اندور اتساعديني، منه المسؤول عن الخدمات اللوجستية ؟ دائما انشوفو يغير ما نعرف اسمو\nFriendly Colleague: I am well, thank God. That is Mr. Brahim

## Sampling hyperparameter sweep

Grid over `sampling_temperature` and `top_p` to see how they affect MBR's
selected output on one representative input - useful for picking defaults
before a full dev-set trial.

In [19]:
temperatures = [0.5, 0.7, 0.9, 1.1]
top_ps = [0.85, 0.9, 0.95, 1.0]

grid_configs = [replace(CFG, sampling_temperature=t, top_p=p) for t, p in itertools.product(temperatures, top_ps)]

messages = [system_msg(), user_msg("Translate the English sentence into Egyptian Arabic (Cairene) Dialect.\n\n### Metadata:\n- Country: EG\n- Domain: Agriculture and farming\n- Participants: Farmer, Buyer\n- Speaker: Farmer\n- Speaker Direction: male -> male\n\n### Conversation History:\nNo previous turns (Start of conversation).\n\n### Sentence to Translate:\nGood morning. The best cucumbers and peppers you'll find today, fresh from the farm.")]

for conf in grid_configs:
    print("temp =", conf.sampling_temperature, "| top_p =", conf.top_p)
    run(messages, conf)
    print("\n" + "=" * 70)

temp = 0.5 | top_p = 0.85

Input: [{'content': 'You are an expert translator.\n\n- Return only the translated text.\n- Do not add any code, explanations, comments, or any other extra text.\n- Keep the meaning and tone and respect the gender direction.\n- Consider the country, the domain, the participants, and the speaker in your translation.\n', 'role': 'system'}, {'content': "Translate the English sentence into Egyptian Arabic (Cairene) Dialect.\n\n### Metadata:\n- Country: EG\n- Domain: Agriculture and farming\n- Participants: Farmer, Buyer\n- Speaker: Farmer\n- Speaker Direction: male -> male\n\n### Conversation History:\nNo previous turns (Start of conversation).\n\n### Sentence to Translate:\nGood morning. The best cucumbers and peppers you'll find today, fresh from the farm.", 'role': 'user'}]

[Beam search, width=5]
  صباح الخير. احسن خيار وفلفل هتلاقيهم النهاردة، طازة من المزرعة.

Sampling 20 candidates (T=0.5, top_p=0.85) ...
Generated 20 candidates (5 unique). Scoring pairwis